# Experiment 03 — Curate Dark-Skin Melanoma Dataset (ITA)

**Goal:** Label all 1,074 real melanoma images (584 ISIC 2020 dermoscopic + 490 Fitzpatrick17k clinical) using our colorimetric ITA estimator, inspect the skin-tone distribution, and export curated dark-skin subsets (`ITA <= 28°`) for StyleGAN2-ADA fine-tuning.

> **Note:** This notebook runs fast on **CPU** (~1–2 minutes) — no GPU needed!

In [ ]:
# 1. Clone project repo (experiment/m500-baseline branch)
import os, sys, subprocess
from pathlib import Path

REPO_ROOT = Path('/content/bias-aware-stylegan')
if not REPO_ROOT.exists():
    subprocess.run([
        'git', 'clone', '-b', 'experiment/m500-baseline',
        'https://github.com/Muneeb-Ahmad-404/bias-aware-stylegan.git',
        str(REPO_ROOT)
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', 'origin', 'experiment/m500-baseline'], check=True)

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print('Working dir:', Path.cwd())

In [ ]:
# 2. Install lightweight dependencies
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'kagglehub', 'pyyaml', 'opencv-python', 'pandas', 'matplotlib', 'pillow'
], check=True)
print('Dependencies installed.')

In [ ]:
# 3. Download 512x512 ISIC 2020 & Fitzpatrick17k datasets from Kaggle
subprocess.run([sys.executable, 'src/data/ingest.py'], check=True)

In [ ]:
# 4. Compute ITA across all 1,074 melanoma images & export curated dark-skin subsets
import json
from src.data.curate_dark_skin import curate_dark_skin

summary = curate_dark_skin(ita_threshold=28.0)
print(json.dumps(summary, indent=2))

In [ ]:
# 5. Plot ISIC 2020 vs Fitzpatrick17k ITA distributions
import pandas as pd
import matplotlib.pyplot as plt
from src.evaluation.ita import ITA_BINS

df = pd.read_csv('data/processed/melanoma_ita_metadata.csv')
order = [b[0] for b in ITA_BINS]
colors = ['#FCE5CD', '#F9CB9C', '#DDA89A', '#C27BA0', '#7F6000', '#4C1130']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (src, title) in zip(axes, [
    ('isic2020', 'ISIC 2020 Melanoma (Dermoscopic)'),
    ('fitzpatrick17k', 'Fitzpatrick17k Melanoma (Clinical)')
]):
    sub = df[df['source'] == src]
    pcts = [(sub['skin_tone_group'] == g).mean() * 100.0 for g in order]
    ax.bar(order, pcts, color=colors)
    ax.set_title(f'{title} (N={len(sub)})')
    ax.set_ylabel('Percentage (%)')
    ax.tick_params(axis='x', rotation=25)

plt.tight_layout()
Path('reports/curation').mkdir(parents=True, exist_ok=True)
plt.savefig('reports/curation/real_melanoma_ita_distribution.png', dpi=150)
plt.show()

In [ ]:
# 6. Visual sanity check: 16 sample curated ISIC dark-skin dermoscopic images (ITA <= 28)
from PIL import Image

sample_paths = sorted(Path('data/processed/finetune_isic_ita28').glob('*.png'))[:16]
print(f'Showing {len(sample_paths)} curated ISIC dark-skin samples:')
if sample_paths:
    cols = 4
    rows = (len(sample_paths) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(10, 2.5 * rows))
    axes = axes.flatten()
    for ax, p in zip(axes, sample_paths):
        ax.imshow(Image.open(p))
        ax.set_title(p.stem.replace('isic2020_', ''), fontsize=8)
        ax.axis('off')
    for ax in axes[len(sample_paths):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()